In [1]:
!pip install -q segmentation_models_pytorch

In [2]:
from pathlib import Path
import pandas as pd 
import matplotlib.pyplot as plt 
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import time

import segmentation_models_pytorch as smp
from torch import nn
from torch.optim import AdamW
import torch
from sklearn.model_selection import train_test_split
import wandb

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
seed = 42
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False 

    os.environ['PYTHONHASHSEED'] = seed 

In [4]:
class CFG:
    loss_fn = nn.BCEWithLogitsLoss()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    loss_fn = smp.losses.SoftBCEWithLogitsLoss()
    num_epochs = 5
    bs = 32
    use_wandb = False
    encoder = "resnet34"

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb")

if CFG.use_wandb:
    wandb.login(key=secret_value_0)
    wandb.init(
        project="ship-detection", 
        config={k:v for k, v in dict(vars(CFG)).items() if '__' not in k},
        group='debug'
    )

In [6]:
def get_mask(labels):
    '''create mask with the help of list of encoded pixels'''
    mask = np.zeros(768 * 768)
    for label in labels:
        label = label.split()
        start = list(map(int, label[::2]))
        run_len = list(map(int, label[1::2]))
    
        for s, r in zip(start, run_len):
            mask[s: s+r] = 1
    mask = mask.reshape(768, 768).T
    return mask

In [18]:
from torch.utils.data import Dataset, DataLoader, Subset
import albumentations as A
import torch

# tfms = A.Compose([
#     A.Resize(height=224, width=224),
#     A.Normalize()])

class ShipData(Dataset):
    def __init__(self, root_dir, train_df, transform=None):
        self.root_dir = root_dir
        self.image_ids = train_df['ImageId'].unique()
        self.tfms = transform
        self.df = train_df

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        labels = df.loc[df['ImageId']==img_id]['EncodedPixels']
        img = Image.open(os.path.join(path/'train_v2', img_id)).resize((224, 224))
        img = np.array(img)
        mask = get_mask(labels)
        if self.tfms:
            result = self.tfms(image=img, mask=mask)
            img, mask = result['image'], result['mask']
        img = torch.tensor(img).permute(2, 0, 1)
        mask = torch.tensor(mask)
        return img, mask

In [19]:
def dice_score(yb, yb_pred, th = 0.5):
    yb = yb.flatten()
    yb_pred = yb_pred.flatten() > th

    total_pixel_matched = (yb == yb_pred).sum()

    dice = (2 * total_pixel_matched) / (2 * len(yb))
    return dice
    
def train_one_epoch(dl, model, optimizer, loss_fn):
    running_loss = 0
    running_acc = 0
    for (xb, yb) in tqdm(dl):
        xb = xb.to(CFG.device)
        yb = yb.to(CFG.device)

        logit = model(xb)
        loss = loss_fn(logit.squeeze(), yb)
        acc = dice_score(yb, logit)
        
        running_loss += loss.item()
        running_acc += acc.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return running_loss / len(dl), running_acc / len(dl)

@torch.no_grad()
def valid_one_epoch(dl, model, optimizer, loss_fn):
    running_loss = 0
    running_acc = 0
    for (xb, yb) in tqdm(dl):
        xb = xb.to(CFG.device)
        yb = yb.to(CFG.device)

        logit = model(xb)
        loss = loss_fn(logit.squeeze(), yb)
        
        acc = dice_score(yb, logit)
        
        running_loss += loss.item()
        running_acc += acc.item()
    return running_loss / len(dl), running_acc / len(dl)

In [20]:
path = Path('/kaggle/input/airbus-ship-detection')
df = pd.read_csv(path/'train_ship_segmentations_v2.csv')
df.fillna('', inplace=True)

train_df, valid_df = train_test_split(df, test_size=0.2)
train_ds = ShipData(path, train_df)
valid_ds = ShipData(path, valid_df)

# train_ds = Subset(train_ds, range(1000))
# valid_ds = Subset(valid_ds, range(100))

train_dl = DataLoader(train_ds, batch_size=CFG.bs, num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=CFG.bs, num_workers=2, pin_memory=True)


model = smp.Unet(
    encoder_name=CFG.encoder,
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
model.to(CFG.device)
lr = 0.001
optimizer = AdamW(lr=lr, params=model.parameters())

# for epoch in range(CFG.num_epochs):
#     tik = time.time()
#     train_loss, train_dice = train_one_epoch(train_dl, model, optimizer, CFG.loss_fn)
#     valid_loss, valid_dice = valid_one_epoch(valid_dl, model, optimizer, CFG.loss_fn)
#     if CFG.use_waTruendb:
#         wandb.log({"Train Loss": train_loss, 
#                        "Valid Loss": valid_loss,
#                        "train dice": train_dice,
#                        "Valid Dice": valid_dice,
#                        "LR":lr })
#     tok = time.time()
#     print(f"ecoch {epoch} | train_loss {train_loss:.4f} | train_dice {train_dice:.4f} | valid_loss {valid_loss:.4f} | valid_dice {valid_dice:.4f} | time {tok-tik:.2f}s")

In [ ]:
print(len(train_dl))
for idx, (xb, yb) in enumerate(train_dl):
    print(idx, xb.shape, yb.shape)

4900
0 torch.Size([32, 3, 224, 224]) torch.Size([32, 768, 768])
1 torch.Size([32, 3, 224, 224]) torch.Size([32, 768, 768])
2 torch.Size([32, 3, 224, 224]) torch.Size([32, 768, 768])
3 torch.Size([32, 3, 224, 224]) torch.Size([32, 768, 768])


In [ ]:
torch.save(model.state_dict(), 'model.pth')
if CFG.use_wandb:
    wandb.save('model.pth')